# Detecção de Defeitos em Concreto — Pipeline de Visão Computacional Clássica

**Dataset:** Corrosion & Spalling — Concrete Defect Segmentation  
**Objetivo:** Segmentar regiões de corrosão e spalling usando técnicas clássicas de VC (sem deep learning).

**Pipeline:**
1. Setup do ambiente e carregamento do dataset
2. Exploração dos dados
3. Pré-processamento
4. Segmentação por thresholding
5. Refinamento morfológico
6. Avaliação (IoU vs máscara ground truth)

## 0. Credenciais (somente Google Colab)

Se estiver no Colab, preencha e execute esta célula antes de qualquer outra.  
**Localmente, pule esta célula** — as credenciais já são lidas do `.env`.

In [ ]:
# ⚠️  COLAB ONLY — skip locally
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_your_token_here"

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # remove esta linha no Colab
from src.setup import setup

DATA_DIR = setup()
DATASET  = os.path.join(DATA_DIR, "spalling_corrosion_patches")
print("Dataset:", DATASET)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

from src.utils import create_run_dir, save_fig, save_metrics

%matplotlib inline
matplotlib.rcParams["figure.figsize"] = (14, 6)

RUN_DIR = create_run_dir()

## 2. Exploração do Dataset

In [ ]:
def load_split(split):
    """Retorna lista de (imagem_path, mask_path) para um split."""
    folder = Path(DATASET) / split
    images = sorted(p for p in folder.iterdir() if not p.stem.endswith("_lab"))
    masks  = [folder / (p.stem + "_lab" + p.suffix) for p in images]
    return list(zip(images, masks))

train_pairs = load_split("train")
val_pairs   = load_split("val")
test_pairs  = load_split("test")

print(f"train: {len(train_pairs)}  val: {len(val_pairs)}  test: {len(test_pairs)}")

In [ ]:
def show_samples(pairs, n=4, title="Amostras"):
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 6))
    for i, (img_path, mask_path) in enumerate(pairs[:n]):
        img  = cv2.cvtColor(cv2.imread(str(img_path)),  cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(img)
        axes[0, i].set_title(img_path.stem)
        axes[0, i].axis("off")
        axes[1, i].imshow(mask, cmap="gray")
        axes[1, i].set_title("mask")
        axes[1, i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    return fig

fig = show_samples(train_pairs, n=4, title="Treino — imagem (cima) e máscara (baixo)")
save_fig("01_dataset_samples", RUN_DIR, fig)

## 3. Pré-processamento

In [ ]:
def preprocess(bgr):
    """Converte para escala de cinza e aplica equalização de histograma."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    eq   = cv2.equalizeHist(gray)
    blur = cv2.GaussianBlur(eq, (5, 5), 0)
    return blur

# Visualizar efeito do pré-processamento
sample_bgr  = cv2.imread(str(train_pairs[0][0]))
sample_pre  = preprocess(sample_bgr)
sample_orig = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (RGB)")
axes[1].imshow(sample_orig, cmap="gray")
axes[1].set_title("Grayscale")
axes[2].imshow(sample_pre, cmap="gray")
axes[2].set_title("Equalizado + Blur")
for ax in axes: ax.axis("off")
plt.tight_layout()
save_fig("02_preprocessing", RUN_DIR)

## 4. Segmentação por Thresholding

In [ ]:
from ipywidgets import interact, IntSlider

def segment(gray, thresh_val):
    _, binary = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY_INV)
    return binary

def show_threshold(thresh_val=80):
    binary = segment(sample_pre, thresh_val)
    gt     = cv2.imread(str(train_pairs[0][1]), cv2.IMREAD_GRAYSCALE)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(sample_pre, cmap="gray")
    axes[0].set_title("Pré-processado")
    axes[1].imshow(binary, cmap="gray")
    axes[1].set_title(f"Threshold = {thresh_val}")
    axes[2].imshow(gt, cmap="gray")
    axes[2].set_title("Ground Truth")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()

interact(show_threshold, thresh_val=IntSlider(min=0, max=255, step=5, value=80))

In [ ]:
# Otsu — threshold automático
def segment_otsu(gray):
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary

otsu_result = segment_otsu(sample_pre)
gt = cv2.imread(str(train_pairs[0][1]), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(otsu_result, cmap="gray")
axes[1].set_title("Otsu")
axes[2].imshow(gt, cmap="gray")
axes[2].set_title("Ground Truth")
for ax in axes: ax.axis("off")
plt.tight_layout()
save_fig("03_otsu_segmentation", RUN_DIR)

## 5. Refinamento Morfológico

In [ ]:
def morphological_refinement(binary, kernel_size=5):
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel)  # remove ruído
    closed  = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)  # fecha buracos
    return closed

refined = morphological_refinement(otsu_result, kernel_size=7)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(otsu_result, cmap="gray")
axes[0].set_title("Otsu")
axes[1].imshow(refined, cmap="gray")
axes[1].set_title("Após morfologia")
axes[2].imshow(gt, cmap="gray")
axes[2].set_title("Ground Truth")
for ax in axes: ax.axis("off")
plt.tight_layout()
save_fig("04_morphological_refinement", RUN_DIR)

## 6. Avaliação — IoU (Intersection over Union)

In [ ]:
def iou(pred, gt):
    """Calcula IoU entre duas máscaras binárias (valores 0 ou 255)."""
    p = pred > 0
    g = gt   > 0
    intersection = np.logical_and(p, g).sum()
    union        = np.logical_or(p,  g).sum()
    return intersection / union if union > 0 else 0.0

def full_pipeline(bgr):
    pre    = preprocess(bgr)
    binary = segment_otsu(pre)
    return morphological_refinement(binary)

# Avaliar sobre todo o conjunto de validação
iou_scores = []
for img_path, mask_path in val_pairs:
    bgr  = cv2.imread(str(img_path))
    gt   = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    pred = full_pipeline(bgr)
    iou_scores.append(iou(pred, gt))

iou_mean = float(np.mean(iou_scores))
iou_min  = float(np.min(iou_scores))
iou_max  = float(np.max(iou_scores))

print(f"IoU médio (val): {iou_mean:.3f}")
print(f"IoU mínimo:      {iou_min:.3f}")
print(f"IoU máximo:      {iou_max:.3f}")

save_metrics(
    {
        "split": "val",
        "n_samples": len(iou_scores),
        "iou_mean": round(iou_mean, 4),
        "iou_min":  round(iou_min,  4),
        "iou_max":  round(iou_max,  4),
    },
    RUN_DIR,
)

plt.figure(figsize=(10, 3))
plt.hist(iou_scores, bins=20, edgecolor="black")
plt.axvline(iou_mean, color="red", linestyle="--", label=f"média = {iou_mean:.3f}")
plt.title("Distribuição de IoU — conjunto de validação")
plt.xlabel("IoU")
plt.ylabel("Frequência")
plt.legend()
plt.tight_layout()
save_fig("05_iou_distribution", RUN_DIR)

In [ ]:
# Visualizar os melhores e piores resultados
sorted_pairs = sorted(zip(iou_scores, val_pairs), key=lambda x: x[0])
worst = [p for _, p in sorted_pairs[:3]]
best  = [p for _, p in sorted_pairs[-3:]]

for label, pairs in [("Piores", worst), ("Melhores", best)]:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    for i, (img_path, mask_path) in enumerate(pairs):
        bgr  = cv2.imread(str(img_path))
        gt   = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        pred = full_pipeline(bgr)
        score = iou(pred, gt)
        axes[i, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        axes[i, 0].set_title("Original")
        axes[i, 1].imshow(pred, cmap="gray")
        axes[i, 1].set_title(f"Predição (IoU={score:.2f})")
        axes[i, 2].imshow(gt, cmap="gray")
        axes[i, 2].set_title("Ground Truth")
        for ax in axes[i]: ax.axis("off")
    fig.suptitle(f"{label} resultados", fontsize=14)
    plt.tight_layout()
    save_fig(f"06_results_{label.lower()}", RUN_DIR, fig)